In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)  # show all columns when previewing

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/brandao/diabetes/description.pdf
/kaggle/input/datasets/brandao/diabetes/diabetic_data.csv
/kaggle/input/datasets/manishasingh07/data-set/IDS_mapping.csv
/kaggle/input/datasets/manishasingh07/data-set/diabetic_data.csv


In [2]:
df = pd.read_csv('/kaggle/input/datasets/brandao/diabetes/diabetic_data.csv')
ids = pd.read_csv('/kaggle/input/datasets/manishasingh07/data-set/IDS_mapping.csv')
ids

,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available
...,...,...
62,22,Transfer from hospital inpt/same fac reslt in...
63,23,Born inside this hospital
64,24,Born outside this hospital
65,25,Transfer from Ambulatory Surgery Center


In [3]:
df.shape

(101766, 50)

101,766 rows and 50 columns, from 130 US hospitals (1999–2008).
Each row is one hospital visit (encounter): there are 101,766 unique encounter_id values.
There are 71,518 unique patients, so some patients appear many times. One patient had 40 visits.

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 101766 entries, 0 to 101765
Data columns (total 50 columns):
 #   Column                    Non-Null Count   Dtype 
---  ------                    --------------   ----- 
 0   encounter_id              101766 non-null  int64 
 1   patient_nbr               101766 non-null  int64 
 2   race                      101766 non-null  object
 3   gender                    101766 non-null  object
 4   age                       101766 non-null  object
 5   weight                    101766 non-null  object
 6   admission_type_id         101766 non-null  int64 
 7   discharge_disposition_id  101766 non-null  int64 
 8   admission_source_id       101766 non-null  int64 
 9   time_in_hospital          101766 non-null  int64 
 10  payer_code                101766 non-null  object
 11  medical_specialty         101766 non-null  object
 12  num_lab_procedures        101766 non-null  int64 
 13  num_procedures            101766 non-null  int64 
 14  num_

diag_1                    101766 non-null  object valuse is in float .
max_glu_serum             5346 non-null    object - has the 5346 entry.
A1Cresult                 17018  has this entry.
Pyers code is object but the value ?

In [5]:
df.head()
df.loc[0]

encounter_id                                 2278392
patient_nbr                                  8222157
race                                       Caucasian
gender                                        Female
age                                           [0-10)
weight                                             ?
admission_type_id                                  6
discharge_disposition_id                          25
admission_source_id                                1
time_in_hospital                                   1
payer_code                                         ?
medical_specialty           Pediatrics-Endocrinology
num_lab_procedures                                41
num_procedures                                     0
num_medications                                    1
number_outpatient                                  0
number_emergency                                   0
number_inpatient                                   0
diag_1                                        

age is in in thformat [0-10)
weight is not presesnt.
time_in_hospital is should in minutes .
Payers code is not present.
Dia2_,dia 3 is not presesnt -' ?'
max_glu_serum,A1Cresult    are NAN

age as [0-10):, it's stored as text ranges, not numbers, so it's an ordered category, and you'll later decide whether to keep it that way or convert it.

weight and payer_code "not present": They are present as columns but filled with "?", which means unknown, and you'll soon measure how much of each column is like that.

time_in_hospital in minutes: This one needs a rethink; check the smallest and largest values in that column and ask yourself whether minutes makes sense for a hospital stay.

diag_2 and diag_3 = "?": Correct for this row; also notice these are diagnosis codes (ICD-9), so even though many look like numbers, they're labels, not quantities.

max_glu_serum and A1Cresult = NaN: This is the important one: the original file contains the word "None" here, meaning the test was not performed, and pandas automatically turned it into NaN, so this NaN is real information, not missing data, and you'll need to handle it differently from "?".

There are 50 columns.

In [6]:
df.loc[0]

encounter_id                                 2278392
patient_nbr                                  8222157
race                                       Caucasian
gender                                        Female
age                                           [0-10)
weight                                             ?
admission_type_id                                  6
discharge_disposition_id                          25
admission_source_id                                1
time_in_hospital                                   1
payer_code                                         ?
medical_specialty           Pediatrics-Endocrinology
num_lab_procedures                                41
num_procedures                                     0
num_medications                                    1
number_outpatient                                  0
number_emergency                                   0
number_inpatient                                   0
diag_1                                        

In [7]:
df ["encounter_id"].value_counts()

encounter_id
443867222    1
2278392      1
149190       1
64410        1
500364       1
            ..
28236        1
15738        1
12522        1
63768        1
55842        1
Name: count, Length: 101766, dtype: int64

: "Each row is one hospital encounter."

In [8]:
df ["patient_nbr"].value_counts()

patient_nbr
88785891     40
43140906     28
88227540     23
1660293      23
23199021     23
             ..
174477542     1
38726739      1
77391171      1
89869032      1
63555939      1
Name: count, Length: 71518, dtype: int64

In [9]:
df.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [10]:
df[["encounter_id","patient_nbr"]].head()

,encounter_id,patient_nbr
0,2278392,8222157
1,149190,55629189
2,64410,86047875
3,500364,82442376
4,16680,42519267


In [11]:
df["patient_nbr"].value_counts().head()

patient_nbr
88785891    40
43140906    28
88227540    23
1660293     23
23199021    23
Name: count, dtype: int64

"Most frequent patient: 40 encounters; repetition is heavy for a few patients."

In [12]:
filt =df["patient_nbr"]== 88785891
df.loc[filt ,["encounter_id","patient_nbr","readmitted"]].value_counts()

encounter_id  patient_nbr  readmitted
119039172     88785891     >30           1
125094312     88785891     <30           1
126171582     88785891     >30           1
137245596     88785891     <30           1
139425576     88785891     <30           1
141994242     88785891     <30           1
150986298     88785891     <30           1
151413846     88785891     <30           1
152188656     88785891     <30           1
153558456     88785891     <30           1
154527444     88785891     >30           1
157363182     88785891     <30           1
158500410     88785891     >30           1
168920160     88785891     >30           1
174689286     88785891     <30           1
175632402     88785891     >30           1
183194856     88785891     >30           1
191345028     88785891     >30           1
205106328     88785891     <30           1
210079272     88785891     >30           1
221588430     88785891     <30           1
223647786     88785891     <30           1
225428574     88

Dataset overview

101,766 rows and 50 columns, from 130 US hospitals (1999–2008).
Each row is one hospital visit (encounter): there are 101,766 unique encounter_id values.
There are 71,518 unique patients, so some patients appear many times. One patient had 40 visits.

Missing data

Missing values are written as "?", not left blank, so pandas does not detect them automatically.
max_glu_serum and A1Cresult show NaN when the test was not done. This is real information, not missing data.
Glucose test done in only [ ]% of visits; A1C test done in only [ ]%.

Column types

age is stored as ranges like [0-10), not exact numbers.
admission_type_id, discharge_disposition_id, and admission_source_id are numbers, but they are codes for categories.
diag_1, diag_2, and diag_3 are diagnosis codes (ICD-9), stored as text.

Target: readmitted

Three values: NO (54,864), >30 (35,545), <30 (11,357).
Readmitted within 30 days: only [ ]%, so the target is imbalanced.
The label comes from hospital records, so some return visits (for example, at other hospitals) are not in the data as rows.

In [13]:
df["readmitted"].value_counts()

readmitted
NO     54864
>30    35545
<30    11357
Name: count, dtype: int64

Some return visits are not in the dataset, so the readmitted label comes from hospital records, not from the rows.

In [14]:
percentage = df["readmitted"].value_counts(normalize=True)*100
percentage

readmitted
NO     53.911916
>30    34.928169
<30    11.159916
Name: proportion, dtype: float64

Readmitted within 30 days: only 11.16%."

In [15]:
filt =(df == "?").sum()
filt

encounter_id                    0
patient_nbr                     0
race                         2273
gender                          0
age                             0
weight                      98569
admission_type_id               0
discharge_disposition_id        0
admission_source_id             0
time_in_hospital                0
payer_code                  40256
medical_specialty           49949
num_lab_procedures              0
num_procedures                  0
num_medications                 0
number_outpatient               0
number_emergency                0
number_inpatient                0
diag_1                         21
diag_2                        358
diag_3                       1423
number_diagnoses                0
max_glu_serum                   0
A1Cresult                       0
metformin                       0
repaglinide                     0
nateglinide                     0
chlorpropamide                  0
glimepiride                     0
acetohexamide 

 percentage_=(df == "?")